### Supplemental code for lesson 1

In [ ]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv")
print(df.head())
print(df.shape)
print(df.info())

In [ ]:
import requests
import pandas as pd
url = "https://api.open-meteo.com/v1/forecast?latitude=25.76&longitude=-80.19&hourly=temperature_2m"
response = requests.get(url)
data = response.json()
temps = data["hourly"]["temperature_2m"]
df = pd.DataFrame(temps, columns=["temperature"])
df.head()


### Kaggle local setup
```bash
# create virtual environment
python -m venv venv 

# activate it (unix based)
source venv/bin/activate

pip install pandas kaggle

# api token generated from kaggle website
export KAGGLE_API_TOKEN=<your_token>

# if configured properly you should see a list of comps
kaggle competitions list
```

In [ ]:
def load_kaggle_dataset(dataset_slug, csv_name):
    import os
    import pandas as pd

    !kaggle datasets download -d {dataset_slug}
    !unzip -o *.zip

    return pd.read_csv(csv_name)

df = load_kaggle_dataset(
    "yasserh/titanic-dataset",
    "Titanic-Dataset.csv"
)
df.head()

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# website to scrape
url = "https://news.ycombinator.com"

# download page
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")
titles = soup.select(".titleline")

data = []

for t in titles:
    data.append(t.text)

df = pd.DataFrame(data, columns=["title"])

df.head()

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin

url = "https://news.ycombinator.com"
response = requests.get(url, timeout=10)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

story_rows = soup.select("tr.athing")

data = []

for row in story_rows:
    title_tag = row.select_one(".titleline > a")
    site_tag = row.select_one(".sitestr")

    subtext_row = row.find_next_sibling("tr")
    subtext = subtext_row.select_one(".subtext") if subtext_row else None

    score_tag = subtext.select_one(".score") if subtext else None
    author_tag = subtext.select_one(".hnuser") if subtext else None
    age_tag = subtext.select_one(".age") if subtext else None

    comment_tag = None
    if subtext:
        links = subtext.select("a")
        if links:
            comment_tag = links[-1]

    data.append({
        "rank": row.select_one(".rank").get_text(strip=True).replace(".", "") if row.select_one(".rank") else None,
        "title": title_tag.get_text(strip=True) if title_tag else None,
        "article_url": title_tag["href"] if title_tag and title_tag.has_attr("href") else None,
        "site": site_tag.get_text(strip=True) if site_tag else None,
        "points": int(score_tag.get_text(strip=True).replace(" points", "").replace(" point", "")) if score_tag else 0,
        "author": author_tag.get_text(strip=True) if author_tag else None,
        "age": age_tag.get_text(strip=True) if age_tag else None,
        "comments": comment_tag.get_text(strip=True) if comment_tag else None,
        "hn_item_id": row.get("id"),
        "hn_comments_url": urljoin(url, f"item?id={row.get('id')}") if row.get("id") else None,
    })

df = pd.DataFrame(data)
df.head(10)